In [0]:
# Divvy Data Lake Project
trip_columns = ["trip_id", "rideable_type", "start_at", "ended_at", "start_station_id", "end_station_id", "rider_id"]
rider_columns = ["rider_id", "first_name", "last_name", "address", "birthday", "account_start_date", "account_end_date", "is_member"]
payment_columns = ["payment_id", "payment_date", "amount", "rider_id"]
station_columns = ["station_id", "station_name", "latitude", "longitude"]

base_path = "/Volumes/databricks_299941/default/divvy_raw_data/"

trip_df = spark.read.csv(base_path + "trips.csv", header=False, inferSchema=True).toDF(*trip_columns)
rider_df = spark.read.csv(base_path + "riders.csv", header=False, inferSchema=True).toDF(*rider_columns)
payment_df = spark.read.csv(base_path + "payments.csv", header=False, inferSchema=True).toDF(*payment_columns)
station_df = spark.read.csv(base_path + "stations.csv", header=False, inferSchema=True).toDF(*station_columns)

trip_df.show(5)
rider_df.show(5)
payment_df.show(5)
station_df.show(5)

+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|         trip_id|rideable_type|           start_at|           ended_at|start_station_id|end_station_id|rider_id|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|89E7AA6C29227EFF| classic_bike|2021-02-12 16:14:56|2021-02-12 16:21:43|             525|           660|   71934|
|0FEFDE2603568365| classic_bike|2021-02-14 17:52:38|2021-02-14 18:12:09|             525|         16806|   47854|
|E6159D746B2DBB91|electric_bike|2021-02-09 19:10:18|2021-02-09 19:19:10|    KA1503000012|  TA1305000029|   70870|
|B32D3199F1C2E75B| classic_bike|2021-02-02 17:49:41|2021-02-02 17:54:06|             637|  TA1305000034|   58974|
|83E463F23575F4BF|electric_bike|2021-02-23 15:07:23|2021-02-23 15:22:37|           13216|  TA1309000055|   39608|
+----------------+-------------+-------------------+-------------------+----------------

In [0]:
## Extract and Load Step
Read CSV files and write them to Bronze Delta tables.
trip_df.write.format("delta").mode("overwrite").saveAsTable("bronze_trip")
rider_df.write.format("delta").mode("overwrite").saveAsTable("bronze_rider")
payment_df.write.format("delta").mode("overwrite").saveAsTable("bronze_payment")
station_df.write.format("delta").mode("overwrite").saveAsTable("bronze_station")

spark.sql("SELECT * FROM bronze_trip LIMIT 5").show()
spark.sql("SELECT * FROM bronze_rider LIMIT 5").show()

+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|         trip_id|rideable_type|           start_at|           ended_at|start_station_id|end_station_id|rider_id|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|D5D9C081468BBE54| classic_bike|2021-07-24 21:32:57|2021-07-24 21:46:53|           13045|  TA1308000050|   25362|
|89C4CD5E85876477| classic_bike|2021-07-09 16:07:05|2021-07-09 16:16:26|           13045|  TA1308000050|   67134|
|A418814FBA2CFE98| classic_bike|2021-07-05 18:05:02|2021-07-05 18:17:01|           13045|  TA1308000050|   21198|
|05243E3DB55EFCE6| classic_bike|2021-07-08 08:03:17|2021-07-08 08:13:51|    TA1309000058|  TA1308000050|   17856|
|F90B3B1D7D92957E| classic_bike|2021-07-01 18:57:09|2021-07-01 19:08:09|           13338|  TA1308000050|   67931|
+----------------+-------------+-------------------+-------------------+----------------

In [0]:
## Transform Step - Dimension Tables
##Create dimension tables for Rider, Station, and Date.
from pyspark.sql.functions import *

spark.sql("DROP TABLE IF EXISTS gold_dim_rider")
spark.sql("DROP TABLE IF EXISTS gold_dim_station")
spark.sql("DROP TABLE IF EXISTS gold_dim_date")

dim_rider = rider_df.select(
    "rider_id",
    "first_name",
    "last_name",
    "birthday",
    "is_member",
    "account_start_date"
)

dim_station = station_df.select(
    "station_id",
    "station_name",
    "latitude",
    "longitude"
)

dim_date = trip_df.select(
    to_date("start_at").alias("full_date")
).distinct() \
 .withColumn("day", dayofmonth("full_date")) \
 .withColumn("month", month("full_date")) \
 .withColumn("quarter", quarter("full_date")) \
 .withColumn("year", year("full_date")) \
 .withColumn("weekday", date_format("full_date", "E")) \
 .withColumn("hour", hour("full_date"))

dim_rider.write.format("delta").mode("overwrite").saveAsTable("gold_dim_rider")
dim_station.write.format("delta").mode("overwrite").saveAsTable("gold_dim_station")
dim_date.write.format("delta").mode("overwrite").saveAsTable("gold_dim_date")

spark.sql("SELECT * FROM gold_dim_rider LIMIT 5").show()
spark.sql("SELECT * FROM gold_dim_station LIMIT 5").show()
spark.sql("SELECT * FROM gold_dim_date LIMIT 5").show()

+--------+----------+---------+----------+---------+------------------+
|rider_id|first_name|last_name|  birthday|is_member|account_start_date|
+--------+----------+---------+----------+---------+------------------+
|   23344|    Curtis|   Harris|1979-12-13|     true|        2019-06-09|
|   23345|    Joshua|   Farmer|1990-07-25|     true|        2020-10-16|
|   23346|      Jill|   Rangel|1988-03-26|     true|        2021-01-10|
|   23347|     Jamie|    Evans|1973-10-10|     true|        2019-01-16|
|   23348|   Shirley|    Gomez|2004-01-22|     true|        2020-02-01|
+--------+----------+---------+----------+---------+------------------+

+------------+--------------------+-----------------+------------------+
|  station_id|        station_name|         latitude|         longitude|
+------------+--------------------+-----------------+------------------+
|         525|Glenwood Ave & To...|        42.012701|-87.66605799999999|
|KA1503000012|  Clark St & Lake St|41.88579466666667|-87.63

In [0]:
## Transform Step - Fact Tables
##Create Trip and Payment fact tables.
fact_trip = trip_df.join(
    rider_df.select("rider_id", "birthday"),
    "rider_id",
    "left"
)

fact_trip = fact_trip.withColumn(
    "duration_minutes",
    (unix_timestamp("ended_at") - unix_timestamp("start_at")) / 60
).withColumn(
    "rider_age_at_trip",
    year(to_date("start_at")) - year("birthday")
).select(
    "trip_id",
    "rider_id",
    "start_station_id",
    "end_station_id",
    "start_at",
    "duration_minutes",
    "rider_age_at_trip"
)

fact_payment = payment_df.join(
    rider_df.select("rider_id", "birthday", "account_start_date"),
    "rider_id",
    "left"
)

fact_payment = fact_payment.withColumn(
    "rider_age_at_account_start",
    year("account_start_date") - year("birthday")
).select(
    "payment_id",
    "rider_id",
    "payment_date",
    "amount",
    "rider_age_at_account_start"
)

fact_trip.write.format("delta").mode("overwrite").saveAsTable("gold_fact_trip")
fact_payment.write.format("delta").mode("overwrite").saveAsTable("gold_fact_payment")

spark.sql("SELECT * FROM gold_fact_trip LIMIT 5").show()
spark.sql("SELECT * FROM gold_fact_payment LIMIT 5").show()

+----------------+--------+----------------+--------------+-------------------+------------------+-----------------+
|         trip_id|rider_id|start_station_id|end_station_id|           start_at|  duration_minutes|rider_age_at_trip|
+----------------+--------+----------------+--------------+-------------------+------------------+-----------------+
|4876DFD9E82828ED|   15536|    TA1308000049|  TA1306000012|2021-09-24 17:41:28|15.316666666666666|               35|
|FE8FBD47BE90B0C6|   75361|    TA1308000049|  TA1306000012|2021-09-29 16:02:33|              6.05|               24|
|D6951DD037CA3B58|   31601|    TA1309000019|  TA1307000134|2021-09-12 13:15:25|15.616666666666667|               25|
|99271B8F68B6811F|   58279|    TA1307000041|  TA1307000134|2021-09-02 17:53:08|10.733333333333333|               47|
|010C7A5DD08F66B0|    2953|    TA1307000041|  TA1307000134|2021-09-01 17:37:48|              5.65|               38|
+----------------+--------+----------------+--------------+-----

In [0]:
## Validation
##Verify generated tables and schema.
spark.sql("SHOW TABLES").show(truncate=False)

spark.sql("DESCRIBE gold_fact_trip").show()
spark.sql("DESCRIBE gold_fact_payment").show()

spark.sql("SELECT COUNT(*) FROM gold_fact_trip").show()
spark.sql("SELECT COUNT(*) FROM gold_fact_payment").show()
spark.sql("SELECT COUNT(*) FROM gold_dim_rider").show()
spark.sql("SELECT COUNT(*) FROM gold_dim_station").show()
spark.sql("SELECT COUNT(*) FROM gold_dim_date").show()

+--------+-----------------+-----------+
|database|tableName        |isTemporary|
+--------+-----------------+-----------+
|default |bronze_payment   |false      |
|default |bronze_rider     |false      |
|default |bronze_station   |false      |
|default |bronze_trip      |false      |
|default |gold_dim_date    |false      |
|default |gold_dim_rider   |false      |
|default |gold_dim_station |false      |
|default |gold_fact_payment|false      |
|default |gold_fact_trip   |false      |
+--------+-----------------+-----------+

+-----------------+---------+-------+
|         col_name|data_type|comment|
+-----------------+---------+-------+
|          trip_id|   string|   NULL|
|         rider_id|      int|   NULL|
| start_station_id|   string|   NULL|
|   end_station_id|   string|   NULL|
|         start_at|timestamp|   NULL|
| duration_minutes|   double|   NULL|
|rider_age_at_trip|      int|   NULL|
+-----------------+---------+-------+

+--------------------+---------+-------+
|     

In [0]:
from pyspark.sql.functions import *

spark.sql("DROP TABLE IF EXISTS gold_fact_trip")

fact_trip = trip_df.join(
    rider_df.select("rider_id", "birthday"),
    "rider_id",
    "left"
)

fact_trip = fact_trip.withColumn(
    "duration_minutes",
    (unix_timestamp("ended_at") - unix_timestamp("start_at")) / 60
)

fact_trip = fact_trip.withColumn(
    "rider_age_at_trip",
    year(to_date("start_at")) - year("birthday")
)

fact_trip = fact_trip.select(
    "trip_id",
    "rider_id",
    "start_station_id",
    "end_station_id",
    "start_at",
    "duration_minutes",
    "rider_age_at_trip"
)

fact_trip.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_trip")

In [0]:
spark.sql("DROP TABLE IF EXISTS gold_fact_payment")

fact_payment = payment_df.join(
    rider_df.select(
        "rider_id",
        "birthday",
        "account_start_date"
    ),
    "rider_id",
    "left"
)

fact_payment = fact_payment.withColumn(
    "rider_age_at_account_start",
    year("account_start_date") - year("birthday")
)

fact_payment = fact_payment.select(
    "payment_id",
    "rider_id",
    "payment_date",
    "amount",
    "rider_age_at_account_start"
)

fact_payment.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_payment")

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

spark.sql("SELECT * FROM gold_fact_trip LIMIT 5").show()

spark.sql("SELECT * FROM gold_fact_payment LIMIT 5").show()

spark.sql("DESCRIBE gold_fact_trip").show()

spark.sql("DESCRIBE gold_fact_payment").show()

+--------+-----------------+-----------+
|database|tableName        |isTemporary|
+--------+-----------------+-----------+
|default |bronze_payment   |false      |
|default |bronze_rider     |false      |
|default |bronze_station   |false      |
|default |bronze_trip      |false      |
|default |gold_dim_date    |false      |
|default |gold_dim_rider   |false      |
|default |gold_dim_station |false      |
|default |gold_fact_payment|false      |
|default |gold_fact_trip   |false      |
+--------+-----------------+-----------+

+----------------+--------+----------------+--------------+-------------------+------------------+-----------------+
|         trip_id|rider_id|start_station_id|end_station_id|           start_at|  duration_minutes|rider_age_at_trip|
+----------------+--------+----------------+--------------+-------------------+------------------+-----------------+
|493E7531A95A143E|    8563|           13053|  TA1307000150|2021-06-02 17:24:19|              27.5|               17